# 03 · Seguridad: homicidios y cárceles

**Fuente:** Ministerio del Interior, catálogo de datos abiertos ([catalogodatos.gub.uy](https://catalogodatos.gub.uy)).

| Dataset | Qué tiene |
|---|---|
| Homicidios dolosos consumados | Una fila por víctima desde 2013: fecha, departamento, motivo aparente, arma, sexo, edad |
| Población privada de libertad, por año, mes y sexo | Personas presas por mes desde 2003 |

## 1. Descarga al volume `raw/seguridad`

In [0]:
import os, urllib.request

RAW = "/Volumes/workspace/pobreza/raw/seguridad"
os.makedirs(RAW, exist_ok=True)

URLS = {
    "homicidios.csv": "https://catalogodatos.gub.uy/dataset/999f2edc-5ef5-4d41-bed7-824a5635ea8d/resource/5ed98add-f127-4377-b529-aa8ad35b77e3/download/homicidios_dolosos_consumados.csv",
    "poblacion_ppl.csv": "https://catalogodatos.gub.uy/dataset/c285b157-a4c0-43b8-ab30-6cca8b54e9e0/resource/bb9a8577-2127-43e2-84bd-b8b3535f18ff/download/poblacion_ppl.csv",
}
for archivo, url in URLS.items():
    urllib.request.urlretrieve(url, f"{RAW}/{archivo}")
    print("Descargado:", archivo, os.path.getsize(f"{RAW}/{archivo}"), "bytes")

## 2. Bronze

In [0]:
def leer_csv(ruta):
    return spark.read.option("header", True).option("encoding", "UTF-8").csv(ruta)

hom_bronze = leer_csv(f"{RAW}/homicidios.csv")
ppl_bronze = leer_csv(f"{RAW}/poblacion_ppl.csv")

for df, nombre in [(hom_bronze, "homicidios_bronze"), (ppl_bronze, "carceles_bronze")]:
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"workspace.pobreza.{nombre}")
    print(nombre, df.count(), "filas")

## 3. Silver

- Columnas en minúscula y sin Ñ.
- Motivos agrupados en 5 categorías ("Otros motivos" y "Sin dato" se juntan).
- La edad viene vacía en algunas filas: `try_cast` la deja en nulo en vez de fallar.

In [0]:
from pyspark.sql import functions as F

MOTIVOS = {
    "AJUSTE DE CUENTAS/ CONFLICTOS ENTRE CRIMINALES": "Conflictos entre criminales",
    "ALTERCADOS ESPONTANEOS/ CONFLICTOS DIVERSOS": "Altercados y conflictos",
    "VIOLENCIA DOMESTICA Y ASOCIADOS": "Violencia doméstica",
    "RAPIÑA / HURTO / COPAMIENTO": "Rapiña o hurto",
}
mapa_motivo = F.create_map([F.lit(x) for kv in MOTIVOS.items() for x in kv])

hom_silver = (spark.table("workspace.pobreza.homicidios_bronze").select(
    F.col("ID_VICTIMA").alias("id_victima"),
    F.to_date("FECHA", "dd.MM.yyyy").alias("fecha"),
    F.col("AÑO").cast("int").alias("anio"),
    F.initcap("DEPARTAMENTO").alias("departamento"),
    F.coalesce(mapa_motivo[F.col("MOTIVO_APARENTE")], F.lit("Otros / sin dato")).alias("motivo"),
    (F.col("ARMAREC") == "ARMA DE FUEGO").alias("arma_fuego"),
    (F.col("ACLARADO") == "ACLARADO").alias("aclarado"),
    F.initcap("SEXO").alias("sexo"),
    F.expr("try_cast(trim(EDADCALC) AS INT)").alias("edad"),
))
hom_silver.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.pobreza.homicidios_silver")

ppl_silver = (spark.table("workspace.pobreza.carceles_bronze").select(
    F.col("AÑO").cast("int").alias("anio"), F.col("MES").alias("mes"),
    F.col("HOMBRE").cast("int").alias("hombres"), F.col("MUJER").cast("int").alias("mujeres"))
  .withColumn("total", F.col("hombres") + F.col("mujeres")))
ppl_silver.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.pobreza.carceles_silver")

display(hom_silver.groupBy("anio").count().orderBy("anio"))

## 4. Gold + export

- Solo años completos (hasta 2025).
- Cárceles: dato de **diciembre** de cada año.

In [0]:
import os

def carpeta_data():
    ruta = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
    if not ruta.startswith("/Workspace"):
        ruta = "/Workspace" + ruta
    if os.path.basename(ruta) == "notebooks":   # si estoy en la carpeta notebooks...
        ruta = os.path.dirname(ruta)            # ...subo un nivel, a la raíz del repo
    os.makedirs(f"{ruta}/data", exist_ok=True)
    return f"{ruta}/data"

def guardar_gold(df, nombre):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"workspace.pobreza.{nombre}_gold")
    df.toPandas().to_json(f"{carpeta_data()}/{nombre}.json", orient="records", force_ascii=False, indent=1)
    print("OK:", nombre)

In [0]:
h = spark.table("workspace.pobreza.homicidios_silver").filter(F.col("anio") <= 2025)

homicidios_por_anio = (h.groupBy("anio").agg(
        F.count("*").alias("n"),
        F.sum(F.col("arma_fuego").cast("int")).alias("n_fuego"),
        F.sum(F.col("aclarado").cast("int")).alias("n_aclarados"))
    .orderBy("anio"))

orden = (F.when(F.col("motivo") == "Conflictos entre criminales", 1)
          .when(F.col("motivo") == "Altercados y conflictos", 2)
          .when(F.col("motivo") == "Violencia doméstica", 3)
          .when(F.col("motivo") == "Rapiña o hurto", 4).otherwise(5))
homicidios_por_motivo = (h.groupBy("anio", "motivo").agg(F.count("*").alias("n"))
    .withColumn("orden", orden).orderBy("anio", "orden"))

carceles_poblacion = (spark.table("workspace.pobreza.carceles_silver")
    .filter(F.col("mes") == "Diciembre").select("anio", "hombres", "mujeres", "total").orderBy("anio"))

guardar_gold(homicidios_por_anio, "homicidios_por_anio")
guardar_gold(homicidios_por_motivo, "homicidios_por_motivo")
guardar_gold(carceles_poblacion, "carceles_poblacion")
display(homicidios_por_anio)

## Hallazgos

- Los homicidios por **conflictos entre criminales** pasaron de 77 en 2013 a 215 en 2025. Hoy son el 57% del total.
- Más de 2 de cada 3 homicidios son con arma de fuego (68% en 2025).
- La población presa pasó de 6.757 (dic. 2003) a 16.107 (dic. 2024). Las mujeres presas se multiplicaron por 4.